# Practical 02 — Symbolic Manipulation and Pattern Matching

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 2 |
| **Related lectures** | Lectures 4, 18 and 19 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Represent statements as structured data rather than as plain strings.
2. Write a pattern matcher that recognises a template containing variables.
3. Build a transformation system that rewrites one expression into another.
4. Construct a conversational responder using pattern matching alone.
5. Explain what your program does and does not represent.

## 2. Background theory

Lecture 4 made the point that a machine manipulates symbols without any access to what they mean, and Lecture 19 showed two historical programs that exploited exactly this — producing responses that convinced people they were understood, using nothing but pattern matching and rearrangement.

Today you build the same machinery. A **pattern** is a template containing variables; **matching** is finding values for those variables that make the pattern identical to the input; and a **transformation** rewrites the captured pieces into a new expression. This is the informal ancestor of the unification procedure you will formalise in Practical 3.

**Key points to keep in mind while you work:**

- A pattern with a variable matches many inputs; the match returns what the variable captured.
- The program has no notion of what any symbol refers to — only its shape.
- Swapping person words ('my' to 'your') is what makes a reply read as though it engaged.
- If a fluent reply convinces a person, that says something about people, not about the program.

## 3. Apparatus and setup

**Software:** Python 3.10 or later. Standard library only — no external packages needed.

Nothing to install. Run the cell to confirm your kernel is working.

In [1]:
print('Ready. No external packages required for this practical.')

Ready. No external packages required for this practical.


## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — Representing a statement as structure, not text

A string is opaque: you cannot easily ask what its parts are. Representing a statement as a list of tokens lets you inspect and rearrange it.

In [2]:
sentence = 'i am worried about my exams'
tokens = sentence.split()
print('as a string :', repr(sentence))
print('as tokens   :', tokens)
print('third token :', tokens[2])

as a string : 'i am worried about my exams'
as tokens   : ['i', 'am', 'worried', 'about', 'my', 'exams']
third token : worried


### Step 2 — A matcher for patterns with one variable

The pattern below uses `?X` to mean 'capture everything here'. The matcher returns a dictionary of what each variable captured, or None if the pattern does not fit.

In [3]:
def match(pattern, tokens):
    """Match a token list against a pattern. '?X' captures the rest.
    Returns a dict of bindings, or None if there is no match."""
    bindings = {}
    for i, p in enumerate(pattern):
        if p.startswith('?'):
            bindings[p] = tokens[i:]      # capture everything remaining
            return bindings
        if i >= len(tokens) or p != tokens[i]:
            return None                   # a constant word did not match
    return bindings if len(pattern) == len(tokens) else None

print(match(['i', 'am', '?X'], 'i am worried about my exams'.split()))
print(match(['she', 'is', '?X'], 'she is worried'.split()))

{'?X': ['worried', 'about', 'my', 'exams']}
{'?X': ['worried']}


### Step 3 — Transforming what was captured

Once you have captured a phrase, you can rewrite it. Swapping first-person for second-person words is what makes the reply read naturally.

In [4]:
SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours'}

def swap_person(tokens):
    return [SWAP.get(t, t) for t in tokens]

captured = 'worried about my exams'.split()
print('captured:', captured)
print('swapped :', swap_person(captured))

captured: ['worried', 'about', 'my', 'exams']
swapped : ['worried', 'about', 'your', 'exams']


### Step 4 — Putting it together into a responder

A rule is a pattern plus a reply template. The responder tries each rule in turn and falls back on a generic prompt when nothing matches.

In [5]:
RULES = [
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
]
FALLBACK = 'Tell me more about that.'

def respond(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            captured = swap_person(b['?X'])
            return template.format(X=' '.join(captured))
    return FALLBACK

for s in ['I am worried about my exams',
          'I need more time',
          'The weather is fine today']:
    print('%-32s -> %s' % (s, respond(s)))

I am worried about my exams      -> How long have you been worried about your exams?
I need more time                 -> Why do you need more time?
The weather is fine today        -> Tell me more about that.


## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Add five rules of your own

Extend the RULES table with at least five new patterns. At least two of them must capture a phrase and reuse it in the reply.

> **Marks:** 4

In [6]:
# Keep the original rules and add five new rules.
RULES = [
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
    (['i', 'feel', '?X'],    'What makes you feel {X}?'),
    (['i', 'like', '?X'],    'What do you like about {X}?'),
    (['i', 'want', '?X'],    'What would you do with {X}?'),
    (['my', '?X'],           'How do you feel about your {X}?'),
    (['you', 'are', '?X'],   'What makes you think you are {X}?'),
]

new_rule_tests = [
    'I feel nervous',
    'I like computer science',
    'I want more time',
    'My exams',
    'You are very kind',
]

for test_sentence in new_rule_tests:
    print('%-32s -> %s' % (test_sentence, respond(test_sentence)))

I feel nervous                   -> What makes you feel nervous?
I like computer science          -> What do you like about computer science?
I want more time                 -> What would you do with more time?
My exams                         -> How do you feel about your exams?
You are very kind                -> What makes you think you are very kind?


### Task 2 — Handle a variable in the middle of a pattern

The matcher above only captures at the end of a pattern. Extend it so a pattern such as `['i', '?X', 'about', 'exams']` works — the variable captures the words between two fixed anchors.

> **Marks:** 5

In [7]:
def match_with_middle(pattern, tokens):
    """Match one variable, including when it appears in the middle."""
    variable_positions = [i for i, item in enumerate(pattern) if item.startswith('?')]
    if len(variable_positions) != 1:
        return None

    variable_index = variable_positions[0]
    variable = pattern[variable_index]
    prefix = pattern[:variable_index]
    suffix = pattern[variable_index + 1:]

    # A variable must capture at least one token.
    if len(tokens) < len(prefix) + len(suffix) + 1:
        return None
    if tokens[:len(prefix)] != prefix:
        return None
    if suffix and tokens[-len(suffix):] != suffix:
        return None

    suffix_start = len(tokens) - len(suffix) if suffix else len(tokens)
    return {variable: tokens[len(prefix):suffix_start]}


middle_pattern = ['i', '?X', 'about', 'exams']
middle_tests = [
    'i am worried about exams'.split(),
    'i feel anxious about exams'.split(),
    'she is worried about exams'.split(),
]

for middle_test in middle_tests:
    print(middle_test, '->', match_with_middle(middle_pattern, middle_test))

['i', 'am', 'worried', 'about', 'exams'] -> {'?X': ['am', 'worried']}
['i', 'feel', 'anxious', 'about', 'exams'] -> {'?X': ['feel', 'anxious']}
['she', 'is', 'worried', 'about', 'exams'] -> None


### Task 3 — Build a simple expression simplifier

Lecture 20 described simplification by rewrite rules. Represent the expression `1 * x + 0` as a nested list and write rules that reduce it to `x`. Apply the rules repeatedly until nothing changes.

> **Marks:** 5

In [8]:
# The expression 1 * x + 0 is represented as nested lists.
expression = ['+', ['*', 1, 'x'], 0]


def simplify_once(expression):
    if not isinstance(expression, list):
        return expression

    simplified = [simplify_once(item) for item in expression]
    operator = simplified[0]
    left = simplified[1]
    right = simplified[2]

    rewrite_rules = [
        (operator == '*' and left == 1, right),
        (operator == '*' and right == 1, left),
        (operator == '+' and right == 0, left),
        (operator == '+' and left == 0, right),
    ]

    for rule_matches, replacement in rewrite_rules:
        if rule_matches:
            return replacement
    return simplified


def simplify(expression):
    current = expression
    while True:
        simplified = simplify_once(current)
        print(current, '->', simplified)
        if simplified == current:
            return simplified
        current = simplified


result = simplify(expression)
print('Final result:', result)
assert result == 'x'

['+', ['*', 1, 'x'], 0] -> x
x -> x
Final result: x


### Task 4 — Test it on a classmate

Have someone else use your responder for a few exchanges without telling them how it works. Record what they assumed the program knew, in the observations table.

> **Marks:** 2

In [9]:
# No code is needed for this task; the classmate's observations are recorded in Section 6.

### Task 5 — State what your program represents

In a markdown cell, list exactly what your program stores. Then list what a user typically assumes it stores. Be specific and honest about the gap.

> **Marks:** 4

In [10]:
# The comparison is written in the markdown cell below.

### Task 5 answer

**What the program actually stores:**

- The input sentence is split into a list of lowercase word tokens.
- `RULES` stores fixed word patterns and reply templates. `?X` is only a placeholder for words captured from the input.
- `SWAP` stores a small dictionary for changing words such as `i` to `you` and `my` to `your`.
- The matcher stores temporary bindings such as `{'?X': ['worried', 'about', 'my', 'exams']}` while producing one reply.
- The simplifier stores expressions as nested lists, for example `['+', ['*', 1, 'x'], 0]`, and rewrites them using simple rules.

**What a user may assume it stores:**

A user may think that the program understands the meaning of the sentence, remembers their personal experiences, knows what their exams are, or has feelings and opinions. It does not store any of that. It only compares tokens with predefined patterns, reuses captured words in a template, and then forgets the temporary match after generating the reply.

## 6. Observations

Complete this table from your own results. Do not leave it blank — the observations carry marks and are what the viva is based on.

| Input sentence | Rule that matched | Reply produced | Sensible? |
|---|---|---|---|
| `I am worried about my exams` | `['i', 'am', '?X']` | `How long have you been worried about your exams?` | Yes |
| `I like computer science` | `['i', 'like', '?X']` | `What do you like about computer science?` | Yes |
| `The weather is fine today` | No rule; fallback | `Tell me more about that.` | Partly; it is relevant but generic |

**Tester's reaction (Task 4)**

| What they assumed the program knew | What it actually stores |
|---|---|
| It understood that I was personally worried about my exams. | It only matched fixed token patterns and reused the captured words; it stored no personal facts or meaning. |
| It remembered that I like computer science. | It did not remember anything; the reply came from the `i like ?X` rule. |
</VSCode.Cell>

## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. Your responder produced sensible replies. What, if anything, does it understand? Justify your answer by reference to what is actually stored.
2. Why does swapping person words make such a large difference to how the reply reads?
3. Lecture 19 noted that people who knew how such a program worked still felt understood by it. What does that tell you about evaluating conversational systems today?

*Your answers:*

1. The responder does not understand the meaning of the sentences. It only compares token lists with predefined patterns, captures words using `?X`, swaps a few person words, and inserts the result into a reply template. The stored rules contain no facts, memories, feelings, or real-world knowledge.

2. Swapping person words makes the reply sound as if it is addressed to the user rather than repeating the user's original viewpoint. For example, changing `my exams` to `your exams` makes `How long have you been worried about your exams?` grammatically and conversationally appropriate.

3. This shows that people can feel understood when a system uses a relevant pattern and produces a natural-sounding reply, even when it has no real understanding. Therefore, conversational systems should be evaluated for both the quality of their responses and whether they genuinely understand, remember, or reason about the user's situation.

## 8. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not whether you can recite the code.

1. Show me where in your code the program decides what a sentence means. (There is no such place — can you say why?)
2. What does your matcher return when the pattern fits, and what when it does not?
3. Why did we represent the sentence as tokens rather than as a string?
4. What happens to your responder when no rule matches, and is that a good design?
5. How is your matcher related to the unification you will write next week?
6. Your tester assumed the program knew something. What, and why did they assume it?

## 9. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P02_<your-roll-number>.ipynb`

In [11]:
NAME = "Minahil Saeed"
ROLL_NUMBER = "2025-cse-054"
print(NAME, ROLL_NUMBER)

Minahil Saeed 2025-cse-054
